In [ ]:
import confnotebook

In [ ]:
from pathlib import Path

source = Path("../examples/RPA-6542")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

In [ ]:
IDX_FILE = 1
NUM_PAGE = 0

In [ ]:
file = files[IDX_FILE]
output_dir = f"../examples/output/{file.stem}"

пайплайн из build_document на одной странице

In [ ]:
from vision_core.loader.pdf_loader import PDFLoader
from vision_core.pipelines.build_document import DocumentBuildPipeline

builder = DocumentBuildPipeline()

img = PDFLoader(pdf_bytes=file.read_bytes()).get_page_image(page_num=NUM_PAGE, dpi=builder.dpi)

aligned_image, alignment_metadata = builder.orientation_preprocessor.process(img, page_number=NUM_PAGE)
image_ocr = builder.image_preprocessor.process(aligned_image, page_number=NUM_PAGE)
ocr_results, mean_confidence = builder._run_ocr(image_ocr)
tables = builder.table_detector.detect_tables(aligned_image, ocr_results, page_number=NUM_PAGE)

builder.cell_text_filler.fill_cells(tables, ocr_results, image=aligned_image, page_number=NUM_PAGE)
filtered_ocr = builder.cell_text_filler.exclude_table_text(ocr_results, tables)

получим медиану межстрочного интервала

In [ ]:
detector = builder.paragraph_detector

median_row_height = detector._median_row_height(filtered_ocr)

print(median_row_height)

удаляем таблицы на изображении по bbox

In [ ]:
import matplotlib.pyplot as plt

preprocessor = builder.paragraph_detector._preprocessor
table_bboxes = [t.bbox for t in tables]
masked = preprocessor._mask_tables(aligned_image, table_bboxes)

plt.figure(figsize=(10, 10))
plt.imshow(masked)
plt.axis("off")
plt.show()

бинаризация изображения перед морфологическими операциями

In [ ]:
binary = preprocessor._binarize(masked)

plt.figure(figsize=(10, 10))
plt.imshow(binary, cmap="gray")
plt.axis("off")
plt.show()

морфологические опреации для получения грубых форм абзацев

In [ ]:
morphed = preprocessor._morphology(binary, median_row_height)

plt.figure(figsize=(10, 10))
plt.imshow(morphed, cmap="gray")
plt.axis("off")
plt.show()

найдем регионы

In [ ]:
import cv2

regions = preprocessor._find_regions(morphed, page_height=img.shape[0])
reg_image = aligned_image.copy()
for roi in regions:
    x, y, w, h = roi.to_tuple()
    cv2.rectangle(reg_image, (x, y), (w, h), (255, 0, 0), 2)

plt.figure(figsize=(10, 10))
plt.imshow(reg_image)
plt.axis("off")
plt.show()

In [ ]:
paragraphs_mapper = detector._map_ocr_to_regions(filtered_ocr, regions, median_row_height * 0.5)

for p in paragraphs_mapper:
    print(f"[{p.id}] - {p.text}")

In [ ]:
page_shape = (aligned_image.shape[0], aligned_image.shape[1])

paragraphs = [p.classify_by_position(page_shape, table_bboxes, median_row_height * 0.5) for p in paragraphs_mapper]


In [ ]:
from vision_core.entities.paragraph import ParagraphType
from vision_core.utils.drawer import Drawer

TYPE_COLORS = {
    ParagraphType.PAGE_HEADER: (255, 165, 0),  # orange
    ParagraphType.PAGE_FOOTER: (255, 165, 0),  # orange
    ParagraphType.SECTION_TITLE: (220, 20, 60),  # crimson
    ParagraphType.TABLE_CAPTION: (148, 0, 211),  # purple
    ParagraphType.BODY_TEXT: (30, 144, 255),  # blue
    ParagraphType.UNKNOWN: (128, 128, 128),  # gray
}
img_debug = aligned_image.copy()
drawer = Drawer(img_debug)
for p in paragraphs:
    color = TYPE_COLORS[p.type]
    drawer.draw_structure(p.bbox.to_tuple(), color=color, width=2, fill=(*color, 40))
    # drawer.draw_text_in_bbox(p.bbox.to_tuple(), text=p.type.name)
    for idx, b in enumerate(p.blobs):
        drawer.draw_structure(b.to_tuple(), color="green", width=1)
        drawer.draw_text_in_bbox(b.to_tuple(), text=str(idx))

display(drawer.to_pil())